# Week 3: N-gram language models

In [23]:
from nltk.corpus import brown
import random
import math
import pandas as pd
from collections import Counter
import numpy as np
eps = np.finfo(float).eps
random.seed(123)

The Brown Corpus comes preprocessed via word tokenization.

In [24]:
dataset = brown.words()
len(dataset)

1161192

In [3]:
dataset

['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', ...]

In [28]:
type(dataset)

nltk.corpus.reader.util.ConcatenatedCorpusView

For the purpose of experimentation, let's create a train/test split of the dataset.

In [25]:
train_data = dataset[:1000000]
test_data = dataset[1000000:]

In [5]:
train_data

['The', 'Fulton', 'County', 'Grand', 'Jury', 'said', ...]

### Train uni-gram language model

Let's now start by implementing a bag-of-words, or our unigram model.

In [26]:
def get_unigram_vocabulary(dataset):
    types = list(set(dataset))
    return types

def unigram_lm(sequence_tokens, vocabulary):
    BoW = {t: 0 for t in vocabulary}
    counts = dict(Counter(sequence_tokens))
    total = sum(counts.values())
    for token in BoW:
        if token in counts:
            BoW[token] = counts[token]/total + eps
        else:
            BoW[token] = eps
    return BoW

Let's fit our unigram model to our dataset!

In [27]:
brown_unigrams = unigram_lm(train_data, get_unigram_vocabulary(dataset))

In [10]:
brown_unigrams["horse"]

7.400000000022204e-05

### Train an bi-gram language model

Now let's write a function that returns a bigram model. The first step is a function that returns the set of possible bigrams in our dataset.

In [33]:
def get_bigram_vocabulary(dataset):
    bigram_types = []
    pad_token = "[PAD]"
    ## TO DO
    newDataSet = [pad_token] + list(dataset) 
    bigram_types = list(set(zip(newDataSet[:-1], newDataSet[1:])))


    ##
    return bigram_types

In [34]:
%%time
bigrams = get_bigram_vocabulary(dataset)
len(bigrams)

CPU times: total: 3.7 s
Wall time: 3.78 s


455268

In [36]:
#bigrams

Now that we have a way to get the set of bigram types lets write the bigram model (don't forget to implement smoothing by adding eps to all probability values):

In [46]:
def bigram_lm(train_data, dataset):
    bigrams = get_bigram_vocabulary(dataset)
    unigrams = get_unigram_vocabulary(dataset)+["[PAD]"]
    bigram_counts = {t: 0 for t in bigrams}
    unigram_counts = {t: 0 for t in unigrams}
    unigram_counts["[PAD]"] = 1
    bigram_probs = dict()
    ## TO DO
    sequence_tokens = ["[PAD]"] + list(train_data)
    bigram_token = list(zip(sequence_tokens[:-1], sequence_tokens[1:]))
    bigram_cnt = Counter(bigram_token)
    unigram_cnt = Counter(sequence_tokens)
    unigram_counts = {t:unigram_cnt.get(t,0)  for t in unigrams}
    #unigram_counts["[PAD]"] = 1
    bigram_counts = {t:bigram_cnt.get(t,0)  for t in bigrams}
    bigram_probs = {t: min(bigram_counts[t]/unigram_counts[t[0]] + eps,1) if bigram_counts[t]> 0 else eps  for t in bigrams}

    ##
    return bigram_probs

Let's fit a bigram model to the brown corpus.

In [47]:
%%time
brown_bigrams = bigram_lm(train_data, dataset)

CPU times: total: 9.97 s
Wall time: 10.1 s


In [48]:
brown_bigrams

{('in', 'utter'): 5.721151095622639e-05,
 ('clean', '.'): 0.06521739130434805,
 (',', 'real'): 0.00014112049674437058,
 ('found', 'proportionate'): 0.002178649237472989,
 ('receiving', 'as'): 0.03448275862068988,
 ('Tax', "Commissioner's"): 0.11111111111111133,
 ('their', 'property'): 0.0004458314757024066,
 ('many', 'tastes'): 0.0011614401858306518,
 ('top', 'of'): 0.31395348837209325,
 ('a', 'balcony'): 5.35217298225299e-05,
 ('shapes', '.'): 0.07692307692307715,
 ('passing', 'down'): 0.018867924528302108,
 ('be', 'represented'): 0.0005177770107009469,
 ('pace', 'as'): 0.027777777777778,
 ('shin', "''"): 1,
 ('the', 'patch'): 1.7969451931938128e-05,
 ('Mitropoulos', 'is'): 0.5000000000000002,
 ('of', 'flexible'): 3.0255355198092067e-05,
 ('Ruling', ','): 0.20000000000000023,
 ('0.5', 'to'): 0.33333333333333354,
 ('screamed', 'triumphantly'): 0.12500000000000022,
 ('my', 'thinking'): 2.220446049250313e-16,
 ('drains', '.'): 0.20000000000000023,
 ('year', 'was'): 0.008116883116883338,


### Train a tri-gram language model

Lets now repeat these steps but for a trigram model.

In [ ]:
def get_trigram_vocabulary(dataset):
    trigram_types = []
    pad_token = "[PAD]"
    ## TO DO

    ##
    return trigram_types

In [ ]:
def trigram_lm(train_data, dataset):
    trigrams = get_trigram_vocabulary(dataset)
    bigrams = get_bigram_vocabulary(dataset)+[("[PAD]","[PAD]")]
    trigram_counts = {t: 0 for t in trigrams}
    bigram_counts = {t: 0 for t in bigrams}
    trigram_probs = dict()
    ## TO DO

    ##
    return trigram_probs

Let's fit a trigram model to the brown corpus.

In [ ]:
brown_trigrams = trigram_lm(train_data, dataset)

### Compare the perplexity of the test data of each model

Which of these models performs best at representing the test data distribution? Write a function that takes a fitted ngram model and a test dataset and returns the perplexity of that dataset. 

Since the probability of the test data is the product of the probabilities of the ngrams which compose it, it is a very small number and we risk running into a floating-point error when trying to compute it. Thus, we should calculate perplexity in log base 2 space. Here is the formula.

$PP(W) = 2^{-\frac{1}{n}\log P(W)}$

In [ ]:
def get_perplexity(ngram_lm, test_data):
    if type(list(ngram_lm.keys())[0]) is tuple :
        ngram_size = len(list(ngram_lm.keys())[0])
    else:
        ngram_size = 1
    perplexity = 0.0
    n = len(test_data)+(ngram_size-1)
    ## TO DO

    ##
    return perplexity

In [ ]:
get_perplexity(brown_trigrams, train_data)

In [ ]:
def compare_perplexity_scores(models, dataset):
    results = [get_perplexity(lm, dataset) for lm in models]
    return results

models = [brown_unigrams, brown_bigrams, brown_trigrams]
train_perplexity = compare_perplexity_scores(models, train_data)
test_perplexity = compare_perplexity_scores(models, test_data)

results = {'models':['unigram','bigram','trigram'],
 'train_perplexity':train_perplexity,
 'test_perplexity':test_perplexity}

df_results = pd.DataFrame(data=results)
df_results

What do you notice about these results? Why might that be?